--------
Create spark session
--------
--------

In [1]:
from pyspark.sql import SparkSession
from dotenv import load_dotenv
import os

# Load env
load_dotenv()

mongo_uri = os.getenv("MONGO_URI")
mongo_db = os.getenv("MONGO_DB")

# Combine into full URI
mongo_connection = f"{mongo_uri}{mongo_db}"

spark = SparkSession.builder \
    .appName("Olist Transformation") \
    .config("spark.mongodb.read.connection.uri", mongo_connection) \
    .config("spark.mongodb.write.connection.uri", mongo_connection) \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

26/04/02 18:04:43 WARN Utils: Your hostname, sMacBookAir.local resolves to a loopback address: 127.0.0.1; using 192.168.0.238 instead (on interface en0)
26/04/02 18:04:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/fruityty/.ivy2/cache
The jars for the packages stored in: /Users/fruityty/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0d5fd153-7b9e-421c-a623-4b3fb18b30e6;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/10.3.0/mongo-spark-connector_2.12-10.3.0.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.12;10.3.0!mongo-spark-connector_2.12.jar (143ms)
:: resolution report :: resolve 1651ms :: artifacts dl 147ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 from central in [default]
	--------------------

--------
Read data from Mongodb
--------
--------

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("price", DoubleType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_value", DoubleType(), True),
])

In [4]:
# Read collections
orders = spark.read.format("mongodb") \
    .option("collection", "orders") \
    .schema(orders_schema) \
    .load()

order_items = spark.read.format("mongodb") \
    .option("collection", "order_items") \
    .schema(order_items_schema) \
    .load()

products = spark.read.format("mongodb") \
    .option("collection", "products") \
    .schema(products_schema) \
    .load()

customers = spark.read.format("mongodb") \
    .option("collection", "customers") \
    .schema(customers_schema) \
    .load()

payments = spark.read.format("mongodb") \
    .option("collection", "payments") \
    .schema(payments_schema) \
    .load()

In [5]:
orders.printSchema()
order_items.printSchema()
products.printSchema()
customers.printSchema()
payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_value: double (nullable = true)



In [6]:
try:
    orders.limit(5).show()
except Exception as e:
    print(str(e))

+--------------------+--------------------+------------------------+
|            order_id|         customer_id|order_purchase_timestamp|
+--------------------+--------------------+------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|     2017-10-02 10:56:33|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|     2018-07-24 20:41:37|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|     2018-08-08 08:38:49|
|949d5b44dbf5de918...|f88197465ea7920ad...|     2017-11-18 19:28:06|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|     2018-02-13 21:18:39|
+--------------------+--------------------+------------------------+



26/04/02 18:05:01 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


--------
Join tables
--------
--------

In [9]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.functions import sum

# Select columns
orders = orders.select("order_id", "customer_id", "order_purchase_timestamp")
order_items = order_items.select("order_id", "product_id", "price")
products = products.select("product_id", "product_category_name")
customers = customers.select("customer_id", "customer_city", "customer_state")
payments = payments.select("order_id", "payment_value")

# Aggregate payment
payments_agg = payments.groupBy("order_id") \
    .agg(sum("payment_value").alias("payment_value"))

# Join tables
df = orders \
    .join(order_items, "order_id") \
    .join(products, "product_id") \
    .join(customers, "customer_id") \
    .join(payments_agg, "order_id")

# Transform
df_clean = df \
    .withColumn("order_date", to_timestamp(col("order_purchase_timestamp"))) \
    .drop("order_purchase_timestamp")

# Show result
df_clean.limit(5).show(5)
df_clean.printSchema()

+--------------------+--------------------+--------------------+-----+---------------------+--------------------+--------------+-------------+-------------------+
|            order_id|         customer_id|          product_id|price|product_category_name|       customer_city|customer_state|payment_value|         order_date|
+--------------------+--------------------+--------------------+-----+---------------------+--------------------+--------------+-------------+-------------------+
|00018f77f2f0320c5...|f6dd3ec061db4e398...|e5f2d52b802189ee6...|239.9|             pet_shop|     santa fe do sul|            SP|       259.83|2017-04-26 10:53:06|
|00042b26cf59d7ce6...|58dbd0b2d70206bf4...|ac6c3623068f30de0...|199.9|   ferramentas_jardim|     varzea paulista|            SP|       218.04|2017-02-04 13:57:51|
|00054e8431b9d7675...|32e2e6ab09e778d99...|8d4f2bb7e93e6710a...| 19.9|            telefonia|          guararapes|            SP|        31.75|2017-12-10 11:53:48|
|0006ec9db01a64e59...|